# 01 · Pedir una página web

**30 minutos.**

Al terminar esta notebook vas a poder:

- pedirle una página a un servidor y saber si te la dio;
- presentarte ante ese servidor en vez de entrar de incógnito;
- leer el `robots.txt` de un sitio y respetarlo;
- guardar el HTML crudo para no volver a pedirlo.

## Qué pasa cuando abrís una página

Tu navegador le manda un **pedido** a un servidor: "dame esta dirección". El
servidor manda una **respuesta**: un número que dice cómo le fue, y el contenido.

`requests` hace exactamente eso, sin navegador en el medio.

In [ ]:
import requests

URL_ODISEA = "https://www.gutenberg.org/cache/epub/58221/pg58221-images.html"

respuesta = requests.get(URL_ODISEA)
respuesta

## El código de estado

El número es lo primero que hay que mirar.

| Código | Quiere decir |
|---|---|
| **200** | Acá está lo que pediste |
| **404** | Esa dirección no existe |
| **403** | Existe, pero no te la doy |
| **429** | Estás pidiendo demasiado rápido |
| **500** | Se rompió algo del lado del servidor |

In [ ]:
print("Estado:  ", respuesta.status_code)
print("Tipo:    ", respuesta.headers["Content-Type"])
print("Tamaño:  ", len(respuesta.text), "caracteres")

### Un 404 no rompe nada, y ese es el problema

Si pedís una dirección que no existe, Python **no** te avisa con un error. Te
devuelve la respuesta igual y el programa sigue. Si no mirás el estado, vas a
terminar guardando una página de error como si fuera un texto de tu corpus.

In [ ]:
rota = requests.get("https://www.gutenberg.org/esta-pagina-no-existe")

print("Estado:", rota.status_code)
print("¿Salió bien?:", rota.ok)

`raise_for_status()` convierte ese silencio en un error explícito. Es una línea y
te ahorra horas de buscar por qué el corpus tiene basura adentro.

In [ ]:
respuesta.raise_for_status()   # con la buena: no pasa nada, sigue de largo

try:
    rota.raise_for_status()    # con la rota: corta acá
except requests.HTTPError as error:
    print("Cortamos a tiempo:", error)

## El contenido

`.text` es el HTML: el mismo que ves si en el navegador hacés
*ver código fuente*.

In [ ]:
html = respuesta.text
print(html[:400])

## Presentarse

Por defecto `requests` se anuncia como `python-requests/2.x`. Es honesto pero
anónimo: quien administra el servidor no sabe quién sos ni a quién escribirle si
algo molesta.

Presentarse es gratis, y es la diferencia entre un pedido y una intrusión.
Poné un mail al que alguien pueda escribirte de verdad.

In [ ]:
CABECERAS = {
    "User-Agent": "LabPLN 2026 - materia de PLN (tu-mail@ejemplo.com)"
}

respuesta = requests.get(URL_ODISEA, headers=CABECERAS, timeout=20)
print("Estado:", respuesta.status_code)

`timeout=20` quiere decir: si el servidor no contesta en 20 segundos, cortamos.
Sin eso, una notebook se puede quedar colgada para siempre esperando a un sitio
caído. **Ponelo siempre.**

## robots.txt

Casi todo sitio publica en `/robots.txt` qué partes acepta que un programa
recorra. No es una ley: es un acuerdo. Pero ignorarlo es la forma más rápida de
que te bloqueen y de quedar mal parado si tu trabajo se publica.

Python trae el lector incluido. No hay que instalar nada.

In [ ]:
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser


def puedo_scrapear(url, agente="*"):
    """Lee el robots.txt del sitio y responde si esa URL está permitida."""
    reglas = RobotFileParser()
    reglas.set_url(urljoin(url, "/robots.txt"))
    reglas.read()
    return reglas.can_fetch(agente, url)

In [ ]:
BUSQUEDA = "https://www.gutenberg.org/ebooks/search/?query=odisea"

print("El libro:   ", puedo_scrapear(URL_ODISEA))
print("La búsqueda:", puedo_scrapear(BUSQUEDA))

Mismo sitio, dos respuestas distintas. Gutenberg dice `Disallow: /ebooks/search`
y tiene sentido: cada búsqueda le cuesta trabajo al servidor, y el catálogo ya lo
publican aparte para descargar.

Ahí está la lección: **antes de scrapear, fijate si no hay una forma más barata de
conseguir lo mismo.**

## Cortesía

Un humano pide una página cada varios segundos. Un `for` pide cincuenta en dos
segundos. Esa diferencia es la que hace que te bloqueen.

Un segundo de espera entre pedidos alcanza y sobra para lo que vamos a hacer hoy.

### La función que vamos a usar el resto de la cursada

Seis líneas. Se identifica, no espera para siempre, avisa si el servidor dijo que
no, y le da un respiro antes del próximo pedido.

In [ ]:
import time


def pedir(url, pausa=1.0):
    """Descarga una página portándose bien."""
    respuesta = requests.get(url, headers=CABECERAS, timeout=20)
    respuesta.raise_for_status()
    time.sleep(pausa)
    return respuesta.text

## Guardar el crudo

Cada vez que corrés la celda de arriba le pedís la página **de nuevo** al servidor.
Diez pruebas son diez descargas del mismo archivo.

Guardalo una vez y trabajá sobre la copia: es más rápido, más respetuoso, y te
deja seguir trabajando aunque se caiga el wifi.

In [ ]:
from pathlib import Path

DATOS = Path("datos")
DATOS.mkdir(exist_ok=True)
ARCHIVO_ODISEA = DATOS / "odisea.html"

if ARCHIVO_ODISEA.exists():
    print("Ya lo tenía guardado. No vuelvo a pedirlo.")
else:
    ARCHIVO_ODISEA.write_text(pedir(URL_ODISEA), encoding="utf-8")
    print("Descargado y guardado.")

html = ARCHIVO_ODISEA.read_text(encoding="utf-8")
print("Caracteres:", len(html))

> **Windows:** ese `encoding="utf-8"` no es decorativo. Sin él, Python abre los
> archivos con la codificación del sistema, que en Windows no es UTF-8, y los
> acentos se rompen. Ponelo **siempre**, al leer y al escribir.

---

## Antes de scrapear, preguntate

| Si el sitio tiene… | Usá eso |
|---|---|
| una API pública | la API |
| un RSS | el RSS |
| un dataset ya publicado | el dataset |
| un `Disallow` que te alcanza | otra fuente |

Scrapear es el último recurso, no el primero. Es más trabajo, se rompe más seguido
y molesta más al otro lado.

## Ejercicio (5 minutos)

Elegí un medio argentino y averiguá dos cosas:

1. ¿Qué dice su `robots.txt` sobre la portada? (probá con `puedo_scrapear`)
2. ¿Publica un RSS?

Anotá las dos respuestas. Las vas a necesitar para la consigna del final.